In [ ]:
# 1. Instalación de librerías necesarias
!pip install openai-whisper vosk pydub

# 2. Descarga del modelo de lenguaje en español para Vosk (Modelo liviano/eficiente)
!wget -q https://alphacephei.com/vosk/models/vosk-model-small-es-0.42.zip
!unzip -q vosk-model-small-es-0.42.zip -d vosk_model

# 3. Descomprimir archivo de audios
import zipfile
import os

zip_path = 'audios.zip'
extract_path = './audios_mp3'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"✅ ¡Éxito! Audios descomprimidos en: {extract_path}")
else:
    print("❌ Error: No se encontró el archivo 'audios.zip' en la raíz de Colab.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 17.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 101.2 MB/s eta 0:00:00
  Created wheel for openai-whisper: filename=openai_whisper-20250625-py3-none-any.whl size=803979 sha256=38ad144ab24b8316224775443e182ecb0bba3e282c4d3073f0f143eb87e5f9bc
  Stored in directory: /root/.cache/pip/wheels/61/d2/20/09ec9bef734d126cba375b15898010b6cc28578d8afdde5869
  Created wheel for srt: filename=srt-3.5.3-py3-none-any.whl size=22427 sha256=a795652ed92d1f46c3be8f96ea67a04509e815e0b4db7fad641c7fd711ad19af
  Stored in directory: /root/.cache/pip/wheels/7e/75/5b/e1d5c3756631e4bda806f6cc9640153b39484bb6f7b0b8def3
Successfully built openai-whisper srt
✅ ¡Éxito! Audios descomprimidos en: ./audios_mp3


In [ ]:
#Configuración del audio
import glob
from pydub import AudioSegment

wav_path = './audios_wav_16k'
os.makedirs(wav_path, exist_ok=True)

# Tomamos una muestra limitada de hasta 50 audios (.mp3)
mp3_files = sorted(glob.glob(os.path.join(extract_path, '**/*.mp3'), recursive=True))[:50]

print(f"Procesando y estandarizando {len(mp3_files)} archivos de audio...")

for file in mp3_files:
    filename = os.path.basename(file).replace('.mp3', '.wav')
    target_file = os.path.join(wav_path, filename)

    # Conversión técnica: Forzar Mono y canales a 16000Hz (Parámetro de configuración acústica)
    sound = AudioSegment.from_mp3(file)
    sound = sound.set_frame_rate(16000).set_channels(1)
    sound.export(target_file, format="wav")

print(f"✅ Curación completada. 50 audios listos en '{wav_path}'")

Procesando y estandarizando 50 archivos de audio...


/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


✅ Curación completada. 50 audios listos en './audios_wav_16k'


In [ ]:
import os
import glob
import time
import pandas as pd
import whisper
from google.colab import drive

# 1. Montar Google Drive para asegurar el guardado permanente
drive.mount('/content/drive')

# 2. Definir rutas del proyecto
audio_folder = './audios_wav_16k'
output_folder = '/content/drive/MyDrive/Proyecto_Fluidez_Lectora'
os.makedirs(output_folder, exist_ok=True)
csv_output_path = os.path.join(output_folder, 'base_para_correccion_manual.csv')

# 3. Cargar el modelo Whisper Medium (Ajustado para alta precisión semántica)
print("Cargando modelo Whisper 'medium' en GPU (esto puede tomar un momento)...")
# El modelo detectará automáticamente si la GPU está activa
model_medium = whisper.load_model("medium")

# 4. Capturar los 50 audios con el formato especificado
# Buscamos todos los archivos que coincidan con la estructura del nombre
all_audios = sorted(glob.glob(os.path.join(audio_folder, 'spontaneous-speech-es-*.wav')))

# Limitamos estrictamente a los primeros 50 encontrados si hay más
muestras_audios = all_audios[:50]
print(f"Se encontraron {len(muestras_audios)} archivos válidos para procesar.")

resultados_preliminares = []

# 5. Pipeline de Inferencia
print("\nIniciando la generación del Pseudo-Ground Truth...")
for idx, file_path in enumerate(muestras_audios):
    filename = os.path.basename(file_path)

    start_time = time.time()

    # Inferencia con parámetros de generación óptimos para español
    # temperature=0.0 asegura determinismo y reduce alucinaciones léxicas
    result = model_medium.transcribe(
        file_path,
        language='es',
        temperature=0.0,
        beam_size=5 # Incrementamos el tamaño de búsqueda para máxima precisión
    )

    latency = time.time() - start_time
    transcripcion_cruda = result['text'].strip()

    # Almacenamos el nombre del audio y el texto generado
    resultados_preliminares.append({
        'ID_Audio': filename,
        'Texto_Whisper_Medium': transcripcion_cruda,
        'Texto_Corregido_Manual': transcripcion_cruda, # Duplicamos la columna para que edites sobre esta
        'Latencia_Medium_Sec': round(latency, 3)
    })

    print(f"[{idx+1}/50] Procesado: {filename} | Tiempo: {round(latency, 2)}s")

# 6. Guardar en Google Drive
df_correccion = pd.DataFrame(resultados_preliminares)
df_correccion.to_csv(csv_output_path, index=False, encoding='utf-8-sig')

print(f"\n✅ ¡Proceso completado con éxito!")
print(f"📁 El archivo ha sido guardado en tu Drive: {csv_output_path}")
print("💡 Tip: Puedes descargar este CSV, abrirlo en Excel para corregir los errores escuchando el audio, y esa será tu columna 'Ground Truth' definitiva.")

Mounted at /content/drive
Cargando modelo Whisper 'medium' en GPU (esto puede tomar un momento)...


100%|█████████████████████████████████████| 1.42G/1.42G [00:19<00:00, 77.0MiB/s]


Se encontraron 50 archivos válidos para procesar.

Iniciando la generación del Pseudo-Ground Truth...
[1/50] Procesado: spontaneous-speech-es-71834.wav | Tiempo: 4.54s
[2/50] Procesado: spontaneous-speech-es-71835.wav | Tiempo: 2.91s
[3/50] Procesado: spontaneous-speech-es-74952.wav | Tiempo: 1.39s
[4/50] Procesado: spontaneous-speech-es-74953.wav | Tiempo: 1.23s
[5/50] Procesado: spontaneous-speech-es-74954.wav | Tiempo: 1.21s
[6/50] Procesado: spontaneous-speech-es-74955.wav | Tiempo: 1.21s
[7/50] Procesado: spontaneous-speech-es-74956.wav | Tiempo: 1.35s
[8/50] Procesado: spontaneous-speech-es-74957.wav | Tiempo: 1.74s
[9/50] Procesado: spontaneous-speech-es-74958.wav | Tiempo: 1.46s
[10/50] Procesado: spontaneous-speech-es-74959.wav | Tiempo: 1.84s
[11/50] Procesado: spontaneous-speech-es-74960.wav | Tiempo: 4.51s
[12/50] Procesado: spontaneous-speech-es-78185.wav | Tiempo: 0.69s
[13/50] Procesado: spontaneous-speech-es-79577.wav | Tiempo: 1.01s
[14/50] Procesado: spontaneous-speec

In [ ]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 58.3 MB/s eta 0:00:00


In [ ]:
import os
import time
import json
import re
import pandas as pd
import whisper
from vosk import Model, KaldiRecognizer
from jiwer import wer
from google.colab import drive

# 1. Asegurar conexión a Google Drive
drive.mount('/content/drive')

# 2. Definir rutas de archivos
ruta_drive = '/content/drive/MyDrive/Proyecto_Fluidez_Lectora'
csv_corregido_path = os.path.join(ruta_drive, 'base_para_correccion_manual.csv')
csv_resultados_finales = os.path.join(ruta_drive, 'resultados_finales_experimentacion.csv')
audio_folder = './audios_wav_16k'

# Verificar que el archivo corregido exista
if not os.path.exists(csv_corregido_path):
    raise FileNotFoundError(f"❌ No se encontró el archivo corregido en: {csv_corregido_path}. Asegúrate de haberlo subido o guardado bien.")

# 3. Cargar el DataFrame con tus correcciones manuales
df_datos = pd.read_csv(csv_corregido_path)

# 4. Cargar Modelos de Evaluación (Whisper Base y Vosk)
print("Cargando Whisper 'base'...")
model_whisper_base = whisper.load_model("base")

print("Cargando Vosk...")
import glob
vosk_folder = glob.glob('./vosk_model/vosk-model-*')[0]
model_vosk = Model(vosk_folder)

# 5. Función de Normalización (Limpieza de texto para un WER justo)
def normalizar_texto(texto):
    if not isinstance(texto, str):
        return ""
    texto = texto.lower()
    # Eliminar signos de puntuación, exclamación e interrogación
    texto = re.sub(r'[.,\/#!$%\^&\*;:{}=\-_`~()¿?¡""'']', '', texto)
    # Reemplazar múltiples espacios por uno solo y quitar espacios en los extremos
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# 6. Pipeline de Experimentación Colectiva
resultados_finales = []

print("\n🚀 Iniciando evaluación de métricas (WER, Latencia, WCPM) sobre las 50 muestras...")

for idx, row in df_datos.iterrows():
    filename = row['ID_Audio']
    file_path = os.path.join(audio_folder, filename)

    # El texto real es tu columna corregida manualmente
    ground_truth_original = row['Texto_Corregido_Manual']
    ground_truth_limpio = normalizar_texto(ground_truth_original)

    total_palabras_reales = len(ground_truth_limpio.split())

    # Omitir si la fila quedó vacía por error
    if total_palabras_reales == 0:
        continue

    # Supongamos una duración estimada estándar del audio para WCPM (ej. 10 segundos = 0.166 minutos)
    # Si tienes la duración exacta de cada audio en segundos, reemplaza el 10 por esa variable.
    duracion_minutos = 10 / 60

    # ------------------------------------------
    # EVALUACIÓN: WHISPER BASE
    # ------------------------------------------
    start_w = time.time()
    res_w = model_whisper_base.transcribe(file_path, language='es', temperature=0.0, beam_size=3)
    lat_whisper = time.time() - start_w

    txt_w_limpio = normalizar_texto(res_w['text'])
    wer_whisper = wer(ground_truth_limpio, txt_w_limpio)

    # Cálculo de WCPM para Whisper
    palabras_correctas_w = max(0, total_palabras_reales * (1 - wer_whisper))
    wcpm_whisper = palabras_correctas_w / duracion_minutos

    # ------------------------------------------
    # EVALUACIÓN: VOSK
    # ------------------------------------------
    start_v = time.time()
    wf = open(file_path, "rb")
    wf.read(44) # Saltar cabecera WAV
    rec = KaldiRecognizer(model_vosk, 16000)
    txt_vosk_list = []
    while True:
        data = wf.read(4000)
        if len(data) == 0: break
        if rec.AcceptWaveform(data):
            res = json.loads(rec.Result())
            if 'text' in res: txt_vosk_list.append(res['text'])
    final_res = json.loads(rec.FinalResult())
    if 'text' in final_res: txt_vosk_list.append(final_res['text'])
    wf.close()

    lat_vosk = time.time() - start_v
    txt_vosk_limpio = normalizar_texto(" ".join(txt_vosk_list))
    wer_vosk = wer(ground_truth_limpio, txt_vosk_limpio)

    # Cálculo de WCPM para Vosk
    palabras_correctas_v = max(0, total_palabras_reales * (1 - wer_vosk))
    wcpm_vosk = palabras_correctas_v / duracion_minutos

    # ------------------------------------------
    # RECOPILACIÓN DE MÉTRICAS
    # ------------------------------------------
    resultados_finales.append({
        'ID_Audio': filename,
        'Ground_Truth': ground_truth_limpio,
        'Transcripcion_Whisper': txt_w_limpio,
        'Transcripcion_Vosk': txt_vosk_limpio,
        'WER_Whisper': round(wer_whisper, 4),
        'WER_Vosk': round(wer_vosk, 4),
        'Latencia_Whisper_Sec': round(lat_whisper, 3),
        'Latencia_Vosk_Sec': round(lat_vosk, 3),
        'WCPM_Whisper': round(wcpm_whisper, 2),
        'WCPM_Vosk': round(wcpm_vosk, 2)
    })

    if (idx + 1) % 10 == 0:
        print(f"Progreso: {idx + 1}/50 audios analizados.")

# 7. Exportar resultados finales comprimidos a Drive
df_final = pd.DataFrame(resultados_finales)
df_final.to_csv(csv_resultados_finales, index=False, encoding='utf-8-sig')

print(f"\n✅ ¡Fase de experimentación terminada con éxito!")
print(f"📊 Archivo final guardado en Drive: {csv_resultados_finales}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Cargando Whisper 'base'...
Cargando Vosk...

🚀 Iniciando evaluación de métricas (WER, Latencia, WCPM) sobre las 50 muestras...
Progreso: 10/50 audios analizados.
Progreso: 20/50 audios analizados.
Progreso: 30/50 audios analizados.
Progreso: 40/50 audios analizados.
Progreso: 50/50 audios analizados.

✅ ¡Fase de experimentación terminada con éxito!
📊 Archivo final guardado en Drive: /content/drive/MyDrive/Proyecto_Fluidez_Lectora/resultados_finales_experimentacion.csv


In [1]:
import os
import pandas as pd
from google.colab import drive

# 1. Asegurar la conexión a Google Drive
drive.mount('/content/drive')

# 2. Definir la ruta donde se guardó tu archivo final de la celda anterior
ruta_drive = '/content/drive/MyDrive/Proyecto_Fluidez_Lectora'
csv_resultados_finales = os.path.join(ruta_drive, 'resultados_finales_experimentacion.csv')

# Verificar si el archivo realmente existe
if not os.path.exists(csv_resultados_finales):
    raise FileNotFoundError(f"❌ No se encontró el archivo de resultados en: {csv_resultados_finales}")

# 3. Cargar el DataFrame con las métricas obtenidas
df_final = pd.read_csv(csv_resultados_finales)

print("📊 --- RESUMEN ESTADÍSTICO DE LOS MODELOS --- \n")

# 4. Cálculo de Promedios Generales para WHISPER BASE
promedio_wer_whisper = df_final['WER_Whisper'].mean()
promedio_lat_whisper = df_final['Latencia_Whisper_Sec'].mean()
promedio_wcpm_whisper = df_final['WCPM_Whisper'].mean()

desviacion_wer_whisper = df_final['WER_Whisper'].std()

# 5. Cálculo de Promedios Generales para VOSK
promedio_wer_vosk = df_final['WER_Vosk'].mean()
promedio_lat_vosk = df_final['Latencia_Vosk_Sec'].mean()
promedio_wcpm_vosk = df_final['WCPM_Vosk'].mean()

desviacion_wer_vosk = df_final['WER_Vosk'].std()

# =========================================================
# 6. PRESENTACIÓN DE RESULTADOS FORMALES
# =========================================================
print(f"=== MODELO: WHISPER BASE ===")
print(f"• Promedio WER (Tasa Error Palabra): {promedio_wer_whisper:.4f} (± {desviacion_wer_whisper:.4f})")
print(f"• Promedio Latencia Temporal     : {promedio_lat_whisper:.3f} segundos")
print(f"• Promedio WCPM: {promedio_wcpm_whisper:.2f} palabras correctas por minuto\n")

print(f"=== MODELO: VOSK ===")
print(f"• Promedio WER (Tasa Error Palabra): {promedio_wer_vosk:.4f} (± {desviacion_wer_vosk:.4f})")
print(f"• Promedio Latencia Temporal     : {promedio_lat_vosk:.3f} segundos")
print(f"• Promedio WCPM : {promedio_wcpm_vosk:.2f} palabras correctas por minuto\n")

# =========================================================
# 7. TABLA COMPARATIVA DIRECTA (Ideal para copiar a tu Word)
# =========================================================
tabla_comparativa = pd.DataFrame({
    'Métrica': ['WER (Error ↓)', 'Latencia (Segundos ↓)', 'WCPM (Fluidez ↑)'],
    'Whisper Base': [f"{promedio_wer_whisper:.4f}", f"{promedio_lat_whisper:.3f} s", f"{promedio_wcpm_whisper:.2f}"],
    'Vosk': [f"{promedio_wer_vosk:.4f}", f"{promedio_lat_vosk:.3f} s", f"{promedio_wcpm_vosk:.2f}"]
})

print("=== TABLA RESUMEN PARA INFORME TÉCNICO ===")
print(tabla_comparativa.to_string(index=False))

Mounted at /content/drive
📊 --- RESUMEN ESTADÍSTICO DE LOS MODELOS --- 

=== MODELO: WHISPER BASE ===
• Promedio WER (Tasa Error Palabra): 0.3878 (± 0.3934)
• Promedio Latencia Temporal     : 0.421 segundos
• Promedio WCPM: 47.40 palabras correctas por minuto

=== MODELO: VOSK ===
• Promedio WER (Tasa Error Palabra): 0.2891 (± 0.2758)
• Promedio Latencia Temporal     : 0.750 segundos
• Promedio WCPM : 46.08 palabras correctas por minuto

=== TABLA RESUMEN PARA INFORME TÉCNICO ===
              Métrica Whisper Base    Vosk
        WER (Error ↓)       0.3878  0.2891
Latencia (Segundos ↓)      0.421 s 0.750 s
     WCPM (Fluidez ↑)        47.40   46.08
